In [1]:
import torch
import kornia.feature as KF
import os
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt

HPATCHES_PATH = "../../archive/hpatches-sequences-release"

In [2]:
# Check the layers of the LoFTR model
teacher = KF.LoFTR(pretrained='outdoor').eval()
for name, module in teacher.named_modules():
    print(name)


backbone
backbone.conv1
backbone.bn1
backbone.relu
backbone.layer1
backbone.layer1.0
backbone.layer1.0.conv1
backbone.layer1.0.conv2
backbone.layer1.0.bn1
backbone.layer1.0.bn2
backbone.layer1.0.relu
backbone.layer1.1
backbone.layer1.1.conv1
backbone.layer1.1.conv2
backbone.layer1.1.bn1
backbone.layer1.1.bn2
backbone.layer1.1.relu
backbone.layer2
backbone.layer2.0
backbone.layer2.0.conv1
backbone.layer2.0.conv2
backbone.layer2.0.bn1
backbone.layer2.0.bn2
backbone.layer2.0.relu
backbone.layer2.0.downsample
backbone.layer2.0.downsample.0
backbone.layer2.0.downsample.1
backbone.layer2.1
backbone.layer2.1.conv1
backbone.layer2.1.conv2
backbone.layer2.1.bn1
backbone.layer2.1.bn2
backbone.layer2.1.relu
backbone.layer3
backbone.layer3.0
backbone.layer3.0.conv1
backbone.layer3.0.conv2
backbone.layer3.0.bn1
backbone.layer3.0.bn2
backbone.layer3.0.relu
backbone.layer3.0.downsample
backbone.layer3.0.downsample.0
backbone.layer3.0.downsample.1
backbone.layer3.1
backbone.layer3.1.conv1
backbone.la

In [3]:
# loftr_coarse
# coarse_matching

In [4]:
# Check the outputs of the coarse features and confidence matrix
teacher = KF.LoFTR(pretrained='outdoor').eval().cuda()

# use hooks to capture the outputs of the layers we care about
teacher_cache = {}
teacher.loftr_coarse.register_forward_hook(
    lambda m, inp, out: teacher_cache.update({'coarse_feat': out})
)
teacher.coarse_matching.register_forward_hook(
    lambda m, inp, out: teacher_cache.update({'conf_matrix': out})
)

# create dummy input images
img0 = torch.rand(1, 1, 480, 480).cuda()
img1 = torch.rand(1, 1, 480, 480).cuda()

with torch.no_grad():
    t_out = teacher({"image0": img0, "image1": img1})

# check what we captured in the cache
for key, val in teacher_cache.items():
    if isinstance(val, torch.Tensor):
        print(f"{key}: shape={val.shape}, dtype={val.dtype}")
    elif isinstance(val, (tuple, list)):
        for i, v in enumerate(val):
            if isinstance(v, torch.Tensor):
                print(f"{key}[{i}]: shape={v.shape}")
            else:
                print(f"{key}[{i}]: type={type(v)}")
    elif isinstance(val, dict):
        for k, v in val.items():
            if isinstance(v, torch.Tensor):
                print(f"{key}['{k}']: shape={v.shape}")
    else:
        print(f"{key}: type={type(val)}")

coarse_feat[0]: shape=torch.Size([1, 3600, 256])
coarse_feat[1]: shape=torch.Size([1, 3600, 256])
conf_matrix: type=<class 'NoneType'>


From the above result, we noticed that coarse_feat is valid, since it's a tuple with two tensors inside, which is just the coarse feature after img0 and img1 going through the transformer, shape is (B, 3600, 256) (matches: B, H*W, C), 3600 = 60 * 60, 256 is the shape of LoFTR coarse. 

However, conf_matrix has a shape of None. Since Kornia's coarse_matching does not return conf_matrix directly. We need to calculate conf_matrix from coarse_feat without hook. LofTR's dual-softmax was conducted based on the two featrues and then do softmax. 

In [5]:
# We can see that the coarse features have shape (B, 3600, 256) and the confidence matrix has shape (B, 3600, 3600).
# The confidence matrix is computed from the coarse features, so we can verify that the teacher's confidence matrix is consistent with the coarse features.
# We can calculate the teacher confidence matrix from the coarse features and compare it with the one captured from the coarse_matching layer.
import torch.nn.functional as F
'''
This is the code to calculate the teacher confidence matrix from the coarse features. 
You can run this code to verify that the teacher's confidence matrix is consistent with the coarse features. 
The teacher confidence matrix should be similar to the one captured from the coarse_matching layer, which means our understanding of how the teacher computes the confidence matrix is correct.
'''
teacher.loftr_coarse.register_forward_hook(
    lambda m, inp, out: teacher_cache.update({
        'coarse_feat0': out[0],  # (B, 3600, 256)
        'coarse_feat1': out[1],  # (B, 3600, 256)
    })
)

# forward
with torch.no_grad():
    t_out = teacher({"image0": img0, "image1": img1})

# calculate the teacher confidence matrix
feat0 = F.normalize(teacher_cache['coarse_feat0'], dim=-1)
feat1 = F.normalize(teacher_cache['coarse_feat1'], dim=-1)
teacher_conf = torch.bmm(feat0, feat1.transpose(1, 2)) / 0.1  # temperature=0.1
teacher_conf = F.softmax(teacher_conf, dim=-1) * F.softmax(teacher_conf, dim=-2)
# teacher_conf: (B, 3600, 3600)

In [6]:
# Demo note only (no execution needed)
# Student feature KD alignment idea:
#   student coarse_feat0: (B, 128, 60, 60)
#   flatten + permute -> (B, 3600, 128)
#   linear projection  -> (B, 3600, 256)
#   compare with teacher coarse_feat0 using MSE
#
# Real implementation is already included in the training loop cells below.

In [7]:
# Training loop (cell 1/3): imports + setup + helper builders
import sys
import csv
import argparse
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torch.amp import autocast, GradScaler

# Resolve project root robustly for both cwd=main and cwd=main/notebooks
_cwd = Path.cwd().resolve()
_candidates = [_cwd, _cwd.parent]
PROJECT_ROOT = None
for _p in _candidates:
    if (_p / "dataset.py").exists() and (_p / "models").exists() and (_p / "utils" / "geometry.py").exists():
        PROJECT_ROOT = _p
        break
if PROJECT_ROOT is None:
    raise RuntimeError("Cannot locate project root that contains dataset.py/models/utils/geometry.py")

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "utils"))

from dataset import SyntheticHomographyDataset
from models import StudentHybrid, StudentCNN, StudentHybridConfig, StudentCNNConfig
from losses import DistillationLoss
from geometry import create_ground_truth_from_homography  # pyright: ignore[reportMissingImports]


def parse_args_notebook():
    parser = argparse.ArgumentParser()
    parser.add_argument("--student_type", type=str, default="all", choices=["all", "hybrid", "cnn"])
    parser.add_argument("--hpatches_path", type=str, default=HPATCHES_PATH)
    parser.add_argument("--num_pairs", type=int, default=10000)
    parser.add_argument("--val_ratio", type=float, default=0.1)
    parser.add_argument("--batch_size", type=int, default=4)
    parser.add_argument("--epochs", type=int, default=30)
    parser.add_argument("--lr", type=float, default=3e-4)
    parser.add_argument("--weight_decay", type=float, default=1e-2)
    parser.add_argument("--patch_size", type=int, default=480)
    parser.add_argument("--num_workers", type=int, default=0)
    parser.add_argument("--amp", action="store_true", default=True)
    parser.add_argument("--temperature", type=float, default=0.1)
    parser.add_argument("--output_dir", type=str, default="../checkpoints")
    args, _ = parser.parse_known_args()
    return args


def build_teacher(device):
    teacher = KF.LoFTR(pretrained="outdoor").eval().to(device)
    for p in teacher.parameters():
        p.requires_grad = False

    teacher_cache = {}

    def _coarse_hook(_module, _inp, out):
        teacher_cache["coarse_feat0"] = out[0].detach()  # (B, 3600, 256)
        teacher_cache["coarse_feat1"] = out[1].detach()  # (B, 3600, 256)

    teacher.loftr_coarse.register_forward_hook(_coarse_hook)
    return teacher, teacher_cache


@torch.no_grad()
def get_teacher_targets(teacher, teacher_cache, img0, img1, temperature=0.1):
    _ = teacher({"image0": img0, "image1": img1})

    feat0 = F.normalize(teacher_cache["coarse_feat0"], dim=-1)
    feat1 = F.normalize(teacher_cache["coarse_feat1"], dim=-1)

    sim = torch.bmm(feat0, feat1.transpose(1, 2)) / temperature
    teacher_conf = F.softmax(sim, dim=-1) * F.softmax(sim, dim=-2)

    return {
        "conf_matrix": teacher_conf,
        "teacher_coarse_feat0": teacher_cache["coarse_feat0"],
        "teacher_coarse_feat1": teacher_cache["coarse_feat1"],
    }


def build_dataloaders(args):
    dataset = SyntheticHomographyDataset(
        image_dir=args.hpatches_path,
        num_pairs=args.num_pairs,
        patch_size=args.patch_size,
    )

    val_len = int(len(dataset) * args.val_ratio)
    train_len = len(dataset) - val_len
    split_gen = torch.Generator().manual_seed(42)
    train_set, val_set = random_split(dataset, [train_len, val_len], generator=split_gen)

    train_loader = DataLoader(
        train_set,
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_set,
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=args.num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    return train_loader, val_loader


def make_student(student_kind, device, only_gt=False):
    if student_kind == "hybrid":
        config = StudentHybridConfig()
        model = StudentHybrid(config).to(device)
    elif student_kind == "cnn":
        config = StudentCNNConfig()
        model = StudentCNN(config).to(device)
    else:
        raise ValueError(f"Unsupported student type: {student_kind}")

    # Fine-level KD is intentionally disabled in this project setup.
    config.lambda_fine_kd = 0.0

    if only_gt:
        config.lambda_coarse_kd = 0.0
        config.lambda_feat_kd = 0.0
        config.lambda_gt = 1.0

    feature_projector = nn.Linear(config.coarse_dim, config.teacher_coarse_dim).to(device)
    return model, config, feature_projector


def project_student_feat_for_kd(feat_map, projector):
    feat_seq = feat_map.flatten(2).permute(0, 2, 1)  # (B, 3600, 128)
    return projector(feat_seq)  # (B, 3600, 256)

In [8]:
# Training loop (cell 2/3): epoch routine + per-run trainer
import time

def train_or_val_one_epoch(
    model,
    criterion,
    optimizer,
    scaler,
    teacher,
    teacher_cache,
    feature_projector,
    loader,
    device,
    temperature,
    train_mode=True,
):
    meter = {"total": 0.0, "coarse_kd": 0.0, "feat_kd": 0.0, "gt_sup": 0.0}

    if train_mode:
        model.train()
    else:
        model.eval()

    phase = "train" if train_mode else "val"
    pbar = tqdm(loader, desc=f"{phase}", leave=False)

    context = torch.enable_grad() if train_mode else torch.no_grad()
    with context:
        for img0, img1, H_gt in pbar:
            img0 = img0.to(device, non_blocking=True)
            img1 = img1.to(device, non_blocking=True)
            H_gt = H_gt.to(device, non_blocking=True)

            teacher_targets = get_teacher_targets(teacher, teacher_cache, img0, img1, temperature=temperature)
            gt_conf = create_ground_truth_from_homography(
                H_gt, height=img0.shape[-2], width=img0.shape[-1], coarse_scale=8
            )

            if train_mode:
                optimizer.zero_grad(set_to_none=True)

            with autocast(device_type=device.type, enabled=(scaler.is_enabled() and train_mode)):
                student_data = model({"image0": img0, "image1": img1})

                student_data["coarse_feat0_proj"] = project_student_feat_for_kd(
                    student_data["coarse_feat0"], feature_projector
                )
                student_data["coarse_feat1_proj"] = project_student_feat_for_kd(
                    student_data["coarse_feat1"], feature_projector
                )

                loss_dict = criterion(
                    student_data,
                    {
                        "conf_matrix": teacher_targets["conf_matrix"],
                        "teacher_coarse_feat0": teacher_targets["teacher_coarse_feat0"],
                        "teacher_coarse_feat1": teacher_targets["teacher_coarse_feat1"],
                    },
                    {"gt_conf_matrix": gt_conf},
                )

            if train_mode:
                scaler.scale(loss_dict["total"]).backward()
                scaler.step(optimizer)
                scaler.update()

            for k in meter.keys():
                meter[k] += float(loss_dict[k].detach().item())

            done = max(1, pbar.n)
            pbar.set_postfix({
                "total": f"{meter['total'] / done:.4f}",
                "feat": f"{meter['feat_kd'] / done:.4f}",
            })

    num_batches = max(1, len(loader))
    for k in meter:
        meter[k] /= num_batches
    return meter


def run_one_training(
    run_name,
    student_kind,
    only_gt,
    args,
    device,
    teacher,
    teacher_cache,
    train_loader,
    val_loader,
):
    model, config, feature_projector = make_student(student_kind, device, only_gt=only_gt)
    criterion = DistillationLoss(config).to(device)

    optimizer = torch.optim.AdamW(
        list(model.parameters()) + list(feature_projector.parameters()),
        lr=args.lr,
        weight_decay=args.weight_decay,
    )

    scaler = GradScaler("cuda", enabled=(args.amp and device.type == "cuda"))

    run_dir = Path(args.output_dir) / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    log_path = run_dir / "epoch_losses.csv"

    with open(log_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "epoch",
            "train_total", "train_coarse_kd", "train_feat_kd", "train_gt_sup",
            "val_total", "val_coarse_kd", "val_feat_kd", "val_gt_sup",
        ])

    run_start = time.time()
    for epoch in range(1, args.epochs + 1):
        epoch_start = time.time()

        train_meter = train_or_val_one_epoch(
            model=model,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
            teacher=teacher,
            teacher_cache=teacher_cache,
            feature_projector=feature_projector,
            loader=train_loader,
            device=device,
            temperature=args.temperature,
            train_mode=True,
        )

        val_meter = train_or_val_one_epoch(
            model=model,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
            teacher=teacher,
            teacher_cache=teacher_cache,
            feature_projector=feature_projector,
            loader=val_loader,
            device=device,
            temperature=args.temperature,
            train_mode=False,
        )

        epoch_time = time.time() - epoch_start
        elapsed = time.time() - run_start
        progress = (epoch / args.epochs) * 100.0
        avg_epoch = elapsed / epoch
        eta_sec = avg_epoch * (args.epochs - epoch)
        eta_min = eta_sec / 60.0

        print(
            f"[{run_name}] Epoch {epoch:02d}/{args.epochs} ({progress:.1f}%) | "
            f"epoch={epoch_time/60.0:.2f} min, ETA={eta_min:.1f} min | "
            f"train total={train_meter['total']:.4f}, coarse={train_meter['coarse_kd']:.4f}, "
            f"feat={train_meter['feat_kd']:.4f}, gt={train_meter['gt_sup']:.4f} | "
            f"val total={val_meter['total']:.4f}, coarse={val_meter['coarse_kd']:.4f}, "
            f"feat={val_meter['feat_kd']:.4f}, gt={val_meter['gt_sup']:.4f}"
        )

        with open(log_path, "a", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                epoch,
                train_meter["total"], train_meter["coarse_kd"], train_meter["feat_kd"], train_meter["gt_sup"],
                val_meter["total"], val_meter["coarse_kd"], val_meter["feat_kd"], val_meter["gt_sup"],
            ])

        ckpt_path = run_dir / f"epoch_{epoch:02d}.pth"
        torch.save(
            {
                "epoch": epoch,
                "run_name": run_name,
                "student_kind": student_kind,
                "model_state_dict": model.state_dict(),
                "projector_state_dict": feature_projector.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "config": config.__dict__,
                "train_loss": train_meter,
                "val_loss": val_meter,
            },
            ckpt_path,
        )

    return model

In [9]:
# Training loop (cell 3/4): smoke test only
args = parse_args_notebook()
args.num_pairs = 200
args.epochs = 1
args.student_type = "hybrid"

print(
    f"Smoke test config | student_type={args.student_type}, "
    f"num_pairs={args.num_pairs}, epochs={args.epochs}, batch_size={args.batch_size}"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

train_loader, val_loader = build_dataloaders(args)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

teacher, teacher_cache = build_teacher(device)

run_plan = [("run_smoke_hybrid", "hybrid", False)]
for run_name, student_kind, only_gt in run_plan:
    _ = run_one_training(
        run_name=run_name,
        student_kind=student_kind,
        only_gt=only_gt,
        args=args,
        device=device,
        teacher=teacher,
        teacher_cache=teacher_cache,
        train_loader=train_loader,
        val_loader=val_loader,
    )

print("Smoke test finished.")

Smoke test config | student_type=hybrid, num_pairs=200, epochs=1, batch_size=4
Device: cuda
Train batches: 45, Val batches: 5


[run_smoke_hybrid] Epoch 01/1 (100.0%) | epoch=4.91 min, ETA=0.0 min | train total=8.5286, coarse=3.4093, feat=5.0050, gt=2.6168 | val total=9.9913, coarse=4.3188, feat=4.8789, gt=3.2331
Smoke test finished.


In [ ]:
# Training loop (cell 4/4): practical full training plan
args = parse_args_notebook()

# Keep runtime practical for a 5-day deadline.
args.num_pairs = 1000
args.epochs = 5
args.batch_size = 4
args.student_type = "hybrid"

print(
    f"Full run config | student_type={args.student_type}, "
    f"num_pairs={args.num_pairs}, epochs={args.epochs}, batch_size={args.batch_size}"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

train_loader, val_loader = build_dataloaders(args)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

teacher, teacher_cache = build_teacher(device)

# Run only the key experiment.
run_plan = [("run_full_hybrid", "hybrid", False)]

for run_name, student_kind, only_gt in run_plan:
    _ = run_one_training(
        run_name=run_name,
        student_kind=student_kind,
        only_gt=only_gt,
        args=args,
        device=device,
        teacher=teacher,
        teacher_cache=teacher_cache,
        train_loader=train_loader,
        val_loader=val_loader,
    )

print("Full training run finished.")

## Team Update

Smoke test completed to verify the pipeline wiring.

- Smoke test params: `student_type=hybrid`, `num_pairs=200`, `epochs=1`, `batch_size=4`
- Full run params: `student_type=hybrid`, `num_pairs=1000`, `epochs=5`, `batch_size=4`
- Active losses: `coarse_kd`, `feat_kd`, `gt_sup` (`fine_kd` disabled)
- Verified: dataloader split, teacher hook + dual-softmax target, KD + GT loss flow, checkpoint and epoch logging.

Next: run the full training cell and compare validation loss trends from `epoch_losses.csv`.